# Movie Recommendation System
Dataset: `kagglehub` - parasharmanas/movie-recommendation-system

Provides:
1. Content-based recommendations (genre similarity)
2. Item-based collaborative filtering (rating similarity)
3. Personalized recommendations for a given user

**One notebook, two modes**, controlled by the `RUN_TRAINING` flag below:
- `RUN_TRAINING = True` -- downloads the data, builds everything, and saves it to disk. Do this once (or whenever the data/logic changes).
- `RUN_TRAINING = False` -- skips all of that and just loads the previously saved artifacts, so you can predict without re-downloading or re-processing 25M rows every session.

**Memory strategy (important for free Colab):** ~62k movies and 25M ratings means a naive full `n_movies x n_movies` similarity matrix would need tens of GB of RAM. Every recommender here instead uses `sklearn.neighbors.NearestNeighbors` with cosine distance, computing similarity to one query row at a time -- memory stays flat (`O(n_movies)`), not `O(n_movies^2)`.

In [1]:
# Set to False once you've trained + saved at least once, to skip straight to predictions.
RUN_TRAINING = True

# Where artifacts get saved to / loaded from.
# In Colab, /content is wiped when the runtime disconnects -- mount Drive if you
# want this to survive across sessions, not just across cells in one session:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   ARTIFACT_DIR = "/content/drive/MyDrive/movie_recommender_artifacts"
ARTIFACT_DIR = "/content/movie_recommender_artifacts"

## Part 1: Train & Save
Runs only if `RUN_TRAINING = True`. Downloads the dataset, builds the content-based and collaborative-filtering models, and saves everything to `ARTIFACT_DIR`.

In [2]:
if RUN_TRAINING:
    import os
    import glob
    import gc
    import pandas as pd
    import numpy as np
    from scipy.sparse import csr_matrix
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.neighbors import NearestNeighbors
    from difflib import get_close_matches

    import kagglehub

In [3]:
if RUN_TRAINING:
    path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")
    print("Dataset downloaded to:", path)

    csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    print("CSV files found:", csv_files)

    movies_path = next((f for f in csv_files if "movie" in os.path.basename(f).lower()), None)
    ratings_path = next((f for f in csv_files if "rating" in os.path.basename(f).lower()), None)

    if movies_path is None or ratings_path is None:
        raise FileNotFoundError(
            f"Could not auto-detect movies/ratings CSVs in {csv_files}. "
            "Open the path manually and set movies_path / ratings_path yourself."
        )

    movies = pd.read_csv(
        movies_path,
        dtype={"movieId": "int32"},
    )

    # Only load the columns we actually use, with the smallest dtypes that fit,
    # to cut memory for 25M rows roughly in half vs. pandas' int64/float64 defaults.
    ratings = pd.read_csv(
        ratings_path,
        usecols=["userId", "movieId", "rating"],
        dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    )

    print("\nmovies.csv columns:", list(movies.columns))
    print("ratings.csv columns:", list(ratings.columns))
    print("movies shape:", movies.shape, "| ratings shape:", ratings.shape)

100%|██████████| 165M/165M [00:02<00:00, 82.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1
CSV files found: ['/root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1/movies.csv', '/root/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1/ratings.csv']

movies.csv columns: ['movieId', 'title', 'genres']
ratings.csv columns: ['userId', 'movieId', 'rating']
movies shape: (62423, 3) | ratings shape: (25000095, 3)


In [4]:
if RUN_TRAINING:
    movies["genres"] = movies["genres"].fillna("")
    movies["title"] = movies["title"].astype(str)
    ratings = ratings.dropna(subset=["userId", "movieId", "rating"])
    ratings = ratings.drop_duplicates(subset=["userId", "movieId"])

    gc.collect()

In [5]:
if RUN_TRAINING:
    tfidf = TfidfVectorizer(token_pattern=r"[^|]+")   # genres are '|' separated, e.g. "Action|Comedy"
    genre_matrix = tfidf.fit_transform(movies["genres"])

    # map title -> row index (drop duplicate titles, keep first)
    title_to_idx = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

    content_nn = NearestNeighbors(metric="cosine", algorithm="brute")
    content_nn.fit(genre_matrix)


    def content_recommend(title, top_n=10):
        """Recommend movies similar in genre to `title`. Handles typos via fuzzy matching."""
        if title not in title_to_idx:
            match = get_close_matches(title, movies["title"], n=1, cutoff=0.4)
            if not match:
                return f"No movie found matching '{title}'"
            title = match[0]

        idx = title_to_idx[title]
        # query just this one movie's neighbors -> O(n_movies) memory, not O(n_movies^2)
        _, indices = content_nn.kneighbors(genre_matrix[idx], n_neighbors=top_n + 1)
        result_idx = [i for i in indices[0] if i != idx][:top_n]
        return movies["title"].iloc[result_idx].tolist()

In [6]:
if RUN_TRAINING:
    # Map raw userId/movieId to dense 0..n-1 indices
    user_ids = ratings["userId"].astype("category")
    movie_ids = ratings["movieId"].astype("category")

    user_idx = user_ids.cat.codes.values
    movie_idx = movie_ids.cat.codes.values

    n_users = len(user_ids.cat.categories)
    n_movies = len(movie_ids.cat.categories)

    # Build sparse user-item matrix directly (no fillna(0), no dense pivot)
    sparse_user_item = csr_matrix(
        (ratings["rating"].values, (user_idx, movie_idx)),
        shape=(n_users, n_movies),
        dtype=np.float32,
    )

    # Keep lookup tables to go from raw IDs -> matrix index and back
    user_id_to_idx = dict(zip(user_ids.cat.categories, range(n_users)))
    movie_id_to_idx = dict(zip(movie_ids.cat.categories, range(n_movies)))
    idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}

    # rows = movies, cols = users -- CSR gives efficient single-row slicing,
    # which is exactly what kneighbors() needs per query
    movie_user_matrix = sparse_user_item.T.tocsr()

    del user_ids, movie_ids, user_idx, movie_idx
    gc.collect()

    # Fit once on the sparse (n_movies x n_users) matrix. This does NOT
    # precompute any n_movies x n_movies similarity matrix -- similarities
    # are computed lazily, per query, inside kneighbors().
    item_nn = NearestNeighbors(metric="cosine", algorithm="brute")
    item_nn.fit(movie_user_matrix)


    def collab_recommend(movie_id, top_n=10):
        """Recommend movies rated similarly to `movie_id` by the same users."""
        if movie_id not in movie_id_to_idx:
            return f"movieId {movie_id} not found in ratings data"

        col = movie_id_to_idx[movie_id]
        _, indices = item_nn.kneighbors(movie_user_matrix[col], n_neighbors=top_n + 1)
        rec_ids = [idx_to_movie_id[i] for i in indices[0] if i != col][:top_n]
        return movies[movies["movieId"].isin(rec_ids)]["title"].tolist()


    def recommend_for_user(user_id, top_n=10, neighbors_per_movie=30):
        """Personalized recommendations: for each movie the user rated, look up its
        nearest neighbors -- one movie (one batch) at a time -- and accumulate a
        rating-weighted score per candidate movie, then return the top results.

        Processing one rated movie per `kneighbors()` call keeps memory flat
        (proportional to `neighbors_per_movie`) regardless of catalog size,
        instead of ever materializing a full item-item similarity matrix.
        """
        if user_id not in user_id_to_idx:
            return f"userId {user_id} not found"

        row = user_id_to_idx[user_id]
        user_row = sparse_user_item.getrow(row)
        rated_cols = set(user_row.indices)

        if not rated_cols:
            return "User has no ratings yet; cannot personalize (cold start)."

        scores = {}
        for col, rating_val in zip(user_row.indices, user_row.data):
            distances, indices = item_nn.kneighbors(
                movie_user_matrix[col], n_neighbors=neighbors_per_movie + 1
            )
            similarities = 1 - distances[0]  # cosine similarity = 1 - cosine distance
            for neighbor_col, sim in zip(indices[0], similarities):
                if neighbor_col in rated_cols:
                    continue
                scores[neighbor_col] = scores.get(neighbor_col, 0.0) + sim * rating_val

        if not scores:
            return "No recommendations available (insufficient overlap with other users)."

        top_cols = sorted(scores, key=scores.get, reverse=True)[:top_n]
        top_movie_ids = [idx_to_movie_id[c] for c in top_cols]
        return movies[movies["movieId"].isin(top_movie_ids)]["title"].tolist()

### Save trained artifacts
`NearestNeighbors(algorithm='brute')` doesn't do expensive precomputation when fit -- it just stores a reference to the data -- so what this actually saves you on future runs is the **data download + CSV parsing + sparse matrix construction**, which is the part that eats memory.

In [7]:
if RUN_TRAINING:
    import joblib
    import os

    # NOTE: /content in Colab is wiped when the runtime disconnects.
    # Mount Drive first if you want this to survive across sessions:
    #   from google.colab import drive
    #   drive.mount('/content/drive')
    #   ARTIFACT_DIR = "/content/drive/MyDrive/movie_recommender_artifacts"
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    # Save the two sparse matrices with scipy's native format (much more memory
    # efficient to write/read than pickling them inside a big joblib blob).
    from scipy.sparse import save_npz
    save_npz(os.path.join(ARTIFACT_DIR, "genre_matrix.npz"), genre_matrix)
    save_npz(os.path.join(ARTIFACT_DIR, "sparse_user_item.npz"), sparse_user_item)
    save_npz(os.path.join(ARTIFACT_DIR, "movie_user_matrix.npz"), movie_user_matrix)

    # Everything else (small lookup tables + fitted NearestNeighbors objects,
    # which are cheap since 'fitting' just stores a reference to the data)
    artifacts = {
        "movies": movies,
        "title_to_idx": title_to_idx,
        "user_id_to_idx": user_id_to_idx,
        "movie_id_to_idx": movie_id_to_idx,
        "idx_to_movie_id": idx_to_movie_id,
        "content_nn": content_nn,
        "item_nn": item_nn,
    }

    artifact_path = os.path.join(ARTIFACT_DIR, "recommender_artifacts.joblib")
    joblib.dump(artifacts, artifact_path)
    print("Saved matrices + artifacts to:", ARTIFACT_DIR)

Saved matrices + artifacts to: /content/movie_recommender_artifacts


## Part 2: Load & Predict
Runs only if `RUN_TRAINING = False` (if you just trained above, the objects are already in memory and this is skipped). Loads the saved artifacts -- no `kagglehub`, no CSV re-parsing, no `.fit()` -- so this is safe to run in a fresh, low-memory runtime.

In [8]:
if not RUN_TRAINING:
    import os
    import joblib
    from scipy.sparse import load_npz
    from difflib import get_close_matches

    artifacts = joblib.load(os.path.join(ARTIFACT_DIR, "recommender_artifacts.joblib"))
    movies = artifacts["movies"]
    title_to_idx = artifacts["title_to_idx"]
    user_id_to_idx = artifacts["user_id_to_idx"]
    movie_id_to_idx = artifacts["movie_id_to_idx"]
    idx_to_movie_id = artifacts["idx_to_movie_id"]
    content_nn = artifacts["content_nn"]
    item_nn = artifacts["item_nn"]

    genre_matrix = load_npz(os.path.join(ARTIFACT_DIR, "genre_matrix.npz"))
    sparse_user_item = load_npz(os.path.join(ARTIFACT_DIR, "sparse_user_item.npz"))
    movie_user_matrix = load_npz(os.path.join(ARTIFACT_DIR, "movie_user_matrix.npz"))

    print("Loaded artifacts from:", ARTIFACT_DIR)

## Recommender functions (same logic as training notebook)

In [9]:
def content_recommend(title, top_n=10):
    """Recommend movies similar in genre to `title`. Handles typos via fuzzy matching."""
    if title not in title_to_idx:
        match = get_close_matches(title, movies["title"], n=1, cutoff=0.4)
        if not match:
            return f"No movie found matching '{title}'"
        title = match[0]
    idx = title_to_idx[title]
    _, indices = content_nn.kneighbors(genre_matrix[idx], n_neighbors=top_n + 1)
    result_idx = [i for i in indices[0] if i != idx][:top_n]
    return movies["title"].iloc[result_idx].tolist()


def collab_recommend(movie_id, top_n=10):
    """Recommend movies rated similarly to `movie_id` by the same users."""
    if movie_id not in movie_id_to_idx:
        return f"movieId {movie_id} not found in ratings data"
    col = movie_id_to_idx[movie_id]
    _, indices = item_nn.kneighbors(movie_user_matrix[col], n_neighbors=top_n + 1)
    rec_ids = [idx_to_movie_id[i] for i in indices[0] if i != col][:top_n]
    return movies[movies["movieId"].isin(rec_ids)]["title"].tolist()


def recommend_for_user(user_id, top_n=10, neighbors_per_movie=30):
    """Personalized recommendations, computed one rated movie at a time so
    memory stays flat regardless of catalog size."""
    if user_id not in user_id_to_idx:
        return f"userId {user_id} not found"
    row = user_id_to_idx[user_id]
    user_row = sparse_user_item.getrow(row)
    rated_cols = set(user_row.indices)
    if not rated_cols:
        return "User has no ratings yet; cannot personalize (cold start)."
    scores = {}
    for col, rating_val in zip(user_row.indices, user_row.data):
        distances, indices = item_nn.kneighbors(
            movie_user_matrix[col], n_neighbors=neighbors_per_movie + 1
        )
        similarities = 1 - distances[0]
        for neighbor_col, sim in zip(indices[0], similarities):
            if neighbor_col in rated_cols:
                continue
            scores[neighbor_col] = scores.get(neighbor_col, 0.0) + sim * rating_val
    if not scores:
        return "No recommendations available (insufficient overlap with other users)."
    top_cols = sorted(scores, key=scores.get, reverse=True)[:top_n]
    top_movie_ids = [idx_to_movie_id[c] for c in top_cols]
    return movies[movies["movieId"].isin(top_movie_ids)]["title"].tolist()

## Demo

In [10]:
sample_title = movies["title"].iloc[0]
sample_user = next(iter(user_id_to_idx))
sample_movie_id = next(iter(movie_id_to_idx))

print("--- Content-based recommendations for:", sample_title, "---")
print(content_recommend(sample_title))

--- Content-based recommendations for: Toy Story (1995) ---
['UglyDolls (2019)', 'Brother Bear 2 (2006)', 'Antz (1998)', "Olaf's Frozen Adventure (2017)", 'Wonder Park (2019)', 'Missing Link (2019)', 'Moana (2016)', 'The Magic Crystal (2011)', 'Scooby-Doo! Mask of the Blue Falcon (2012)', 'Wild, The (2006)']


In [11]:
print("--- Users who liked movieId", sample_movie_id, "also liked ---")
print(collab_recommend(sample_movie_id))

--- Users who liked movieId 1 also liked ---
['Star Wars: Episode IV - A New Hope (1977)', 'Forrest Gump (1994)', 'Lion King, The (1994)', 'Jurassic Park (1993)', 'Aladdin (1992)', 'Independence Day (a.k.a. ID4) (1996)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Star Wars: Episode VI - Return of the Jedi (1983)', 'Back to the Future (1985)', 'Toy Story 2 (1999)']


In [12]:
print("--- Personalized picks for user", sample_user, "---")
print(recommend_for_user(sample_user))

--- Personalized picks for user 1 ---
['Truman Show, The (1998)', 'Matrix, The (1999)', 'American Beauty (1999)', 'Fight Club (1999)', 'Memento (2000)', 'Lord of the Rings: The Fellowship of the Ring, The (2001)', 'Kill Bill: Vol. 1 (2003)', 'Lord of the Rings: The Return of the King, The (2003)', 'Kill Bill: Vol. 2 (2004)', "Avventura, L' (Adventure, The) (1960)"]


## Simple UI
A lightweight `ipywidgets` panel that calls the recommender functions already defined above. No re-downloading, re-training, or re-saving anything -- it just talks to whatever is already in memory (from either the training run or the load-from-disk cell).

In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

mode_dropdown = widgets.Dropdown(
    options=[
        ("Content-based (similar genres to a movie)", "content"),
        ("Collaborative (users who liked this movie also liked...)", "collab"),
        ("Personalized (recommendations for a user)", "personal"),
    ],
    description="Mode:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

query_input = widgets.Text(
    description="Movie title:",
    placeholder="e.g. Toy Story (1995)",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

top_n_slider = widgets.IntSlider(
    value=10, min=1, max=25, description="How many:",
    style={"description_width": "initial"},
)

go_button = widgets.Button(description="Get recommendations", button_style="success", icon="search")
output_area = widgets.Output()


def on_mode_change(change):
    mode = change["new"]
    if mode == "content":
        query_input.description = "Movie title:"
        query_input.placeholder = "e.g. Toy Story (1995)"
    elif mode == "collab":
        query_input.description = "Movie ID:"
        query_input.placeholder = "e.g. 1"
    else:
        query_input.description = "User ID:"
        query_input.placeholder = "e.g. 1"


mode_dropdown.observe(on_mode_change, names="value")


def on_click(_):
    with output_area:
        clear_output()
        mode = mode_dropdown.value
        query = query_input.value.strip()
        top_n = top_n_slider.value
        if not query:
            print("Please enter a value first.")
            return
        try:
            if mode == "content":
                result = content_recommend(query, top_n=top_n)
            elif mode == "collab":
                result = collab_recommend(int(query), top_n=top_n)
            else:
                result = recommend_for_user(int(query), top_n=top_n)
        except ValueError:
            print("That field needs a number for this mode.")
            return

        if isinstance(result, str):
            print(result)
        else:
            for i, title in enumerate(result, 1):
                print(f"{i}. {title}")


go_button.on_click(on_click)

display(widgets.VBox([mode_dropdown, query_input, top_n_slider, go_button, output_area]))
